# Module 8 • Large Language Models

# Lesson 47 • LLM Evaluation, Hallucination, and Reliability

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced  
**Estimated study time:** 180–220 minutes  
**Execution target:** CPU only

---

## Scope

This lesson develops a systematic framework for evaluating large language models
beyond a single benchmark score.

The executable core is fully offline and demonstrates:

- task accuracy;
- exact match and token F1;
- factuality and groundedness checks;
- hallucination labeling;
- citation support;
- confidence calibration;
- expected calibration error;
- Brier score;
- robustness to prompt paraphrases;
- abstention and selective prediction;
- reliability curves;
- bootstrap confidence intervals;
- paired model comparison;
- human-evaluation rubrics;
- error taxonomies.

No external model or API is required.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish capability evaluation from reliability evaluation;
- define hallucination, factuality, faithfulness, and groundedness;
- evaluate answers with exact match and token-level F1;
- separate retrieval errors from generation errors;
- score citation support;
- measure confidence calibration;
- compute expected calibration error and Brier score;
- construct reliability diagrams;
- evaluate robustness under paraphrased prompts;
- measure abstention quality;
- apply bootstrap confidence intervals;
- compare models using paired bootstrap analysis;
- design human-evaluation rubrics;
- report limitations and uncertainty.

## Table of Contents

1. Why LLM Evaluation Is Difficult
2. Capability Versus Reliability
3. Hallucination
4. Factuality
5. Faithfulness and Groundedness
6. Correctness Versus Helpfulness
7. Evaluation Levels
8. Offline Evaluation Dataset
9. Exact Match
10. Token-Level F1
11. String Normalization
12. Factuality Labels
13. Groundedness Labels
14. Citation Support
15. Unsupported Claim Detection
16. Hallucination Rate
17. Selective Prediction
18. Confidence Scores
19. Calibration
20. Brier Score
21. Expected Calibration Error
22. Reliability Diagram
23. Confidence Thresholding
24. Risk-Coverage Trade-Off
25. Prompt Robustness
26. Paraphrase Sensitivity
27. Consistency
28. Adversarial Inputs
29. Retrieval Versus Generation Errors
30. Citation Precision
31. Citation Recall
32. Answer Support Rate
33. Bootstrap Confidence Intervals
34. Paired Bootstrap Comparison
35. Per-Category Analysis
36. Error Taxonomy
37. Human Evaluation
38. Inter-Annotator Agreement
39. Rubric Design
40. Automatic Judge Models
41. Benchmark Contamination
42. Dataset Shift
43. Long-Context Reliability
44. Safety and Reliability
45. Multilingual Evaluation
46. Arabic Evaluation
47. Reproducibility
48. Knowledge Check
49. Exercises
50. Summary and Next Lesson

# 1. Why LLM Evaluation Is Difficult

LLMs produce open-ended text. There may be many acceptable answers, and fluency can
hide factual or reasoning errors.

Evaluation therefore requires multiple complementary metrics.

In [ ]:
import math
import platform
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

evaluation_dimensions = pd.DataFrame(
    [
        ("Capability", "Can the model perform the task?"),
        ("Correctness", "Is the answer right?"),
        ("Groundedness", "Is the answer supported by evidence?"),
        ("Calibration", "Does confidence match correctness?"),
        ("Robustness", "Does behavior remain stable under perturbations?"),
        ("Safety", "Does the model avoid harmful behavior?"),
        ("Efficiency", "What are latency and cost?"),
    ],
    columns=["Dimension", "Question"],
)

evaluation_dimensions

# 2. Capability Versus Reliability

Capability asks whether a model can succeed.

Reliability asks whether its outputs can be trusted consistently across examples,
prompts, and conditions.

# 3. Hallucination

Hallucination is generated content that is unsupported, fabricated, or inconsistent
with available evidence.

Hallucination should be defined operationally for a specific task.

# 4. Factuality

Factuality concerns whether statements are true according to an appropriate
reference source or world knowledge.

# 5. Faithfulness and Groundedness

**Faithfulness** asks whether an output accurately reflects its input or source.

**Groundedness** asks whether claims are supported by provided evidence.

In [ ]:
distinction_table = pd.DataFrame(
    [
        ("Factuality", "Is the statement true?"),
        ("Faithfulness", "Does the output preserve the source meaning?"),
        ("Groundedness", "Is the claim supported by supplied evidence?"),
        ("Helpfulness", "Does the answer satisfy the user's task?"),
    ],
    columns=["Concept", "Question"],
)

distinction_table

# 6. Correctness Versus Helpfulness

A response can be helpful but factually wrong, or factually correct but unhelpful.

These dimensions should be scored separately.

# 7. Evaluation Levels

LLM evaluation can occur at:

- token level;
- answer level;
- claim level;
- conversation level;
- system level.

# 8. Offline Evaluation Dataset

The synthetic benchmark contains reference answers, evidence, candidate model
answers, confidence scores, and categories.

In [ ]:
evaluation_data = pd.DataFrame(
    [
        {
            "id": "q1",
            "category": "rag",
            "question": "What does RAG add before generation?",
            "reference": "external evidence retrieval",
            "evidence": "Retrieval-augmented generation retrieves external evidence before generation.",
            "model_a": "It retrieves external evidence before generation.",
            "model_b": "It trains the model on new evidence before generation.",
            "confidence_a": 0.92,
            "confidence_b": 0.86,
        },
        {
            "id": "q2",
            "category": "evaluation",
            "question": "What does recall at k measure?",
            "reference": "whether relevant evidence appears in the top k results",
            "evidence": "Recall at k measures whether relevant evidence appears among the first k retrieved results.",
            "model_a": "It measures whether relevant evidence appears in the top k results.",
            "model_b": "It measures the percentage of retrieved results that are relevant.",
            "confidence_a": 0.89,
            "confidence_b": 0.88,
        },
        {
            "id": "q3",
            "category": "transformers",
            "question": "Why is causal masking used?",
            "reference": "to prevent attention to future tokens",
            "evidence": "Causal masking prevents a decoder from attending to future tokens.",
            "model_a": "It prevents the decoder from attending to future tokens.",
            "model_b": "It removes padding tokens from the vocabulary.",
            "confidence_a": 0.95,
            "confidence_b": 0.81,
        },
        {
            "id": "q4",
            "category": "tokenization",
            "question": "Why can tokenizer efficiency matter?",
            "reference": "it affects effective context length",
            "evidence": "Tokenizer efficiency affects the effective context length available to the model.",
            "model_a": "It affects effective context length.",
            "model_b": "It guarantees better factuality.",
            "confidence_a": 0.80,
            "confidence_b": 0.78,
        },
        {
            "id": "q5",
            "category": "arabic",
            "question": "What should fully vocalized Arabic preserve?",
            "reference": "tashkeel",
            "evidence": "For fully vocalized Arabic tasks, tashkeel should be preserved consistently.",
            "model_a": "It should preserve tashkeel.",
            "model_b": "It should remove tashkeel before tokenization.",
            "confidence_a": 0.94,
            "confidence_b": 0.75,
        },
        {
            "id": "q6",
            "category": "context",
            "question": "What is a context window?",
            "reference": "the maximum token sequence the model can process together",
            "evidence": "A context window limits how many tokens a model can process together.",
            "model_a": "It is the maximum token sequence the model can process together.",
            "model_b": "It is the number of model parameters.",
            "confidence_a": 0.91,
            "confidence_b": 0.70,
        },
        {
            "id": "q7",
            "category": "retrieval",
            "question": "What is hybrid retrieval?",
            "reference": "a combination of sparse and dense retrieval signals",
            "evidence": "Hybrid retrieval combines sparse lexical and dense semantic relevance signals.",
            "model_a": "It combines sparse lexical and dense semantic retrieval signals.",
            "model_b": "It combines two decoder layers.",
            "confidence_a": 0.90,
            "confidence_b": 0.65,
        },
        {
            "id": "q8",
            "category": "calibration",
            "question": "What does calibration compare?",
            "reference": "confidence with observed correctness",
            "evidence": "Calibration compares predicted confidence with observed correctness.",
            "model_a": "It compares model confidence with observed correctness.",
            "model_b": "It compares training and validation dataset sizes.",
            "confidence_a": 0.84,
            "confidence_b": 0.83,
        },
        {
            "id": "q9",
            "category": "evaluation",
            "question": "What does MRR reward?",
            "reference": "ranking the first relevant result near the top",
            "evidence": "Mean reciprocal rank rewards placing the first relevant result near the top.",
            "model_a": "It rewards ranking the first relevant result near the top.",
            "model_b": "It measures generation diversity.",
            "confidence_a": 0.88,
            "confidence_b": 0.74,
        },
        {
            "id": "q10",
            "category": "hallucination",
            "question": "What is hallucination in grounded QA?",
            "reference": "unsupported generated content",
            "evidence": "Hallucination is generated content that is unsupported by available evidence.",
            "model_a": "It is generated content unsupported by the evidence.",
            "model_b": "It is any answer longer than the reference.",
            "confidence_a": 0.86,
            "confidence_b": 0.77,
        },
    ]
)

evaluation_data.head()

# 9. Exact Match

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def exact_match(
    reference: str,
    prediction: str,
) -> float:
    return float(
        normalize_text(reference)
        == normalize_text(prediction)
    )


exact_match(
    "Tashkeel",
    "tashkeel",
)

# 10. Token-Level F1

In [ ]:
def token_f1(
    reference: str,
    prediction: str,
) -> float:
    reference_tokens = normalize_text(
        reference
    ).split()

    prediction_tokens = normalize_text(
        prediction
    ).split()

    if (
        not reference_tokens
        or not prediction_tokens
    ):
        return 0.0

    overlap = sum(
        (
            Counter(reference_tokens)
            & Counter(prediction_tokens)
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = (
        overlap
        / len(prediction_tokens)
    )

    recall = (
        overlap
        / len(reference_tokens)
    )

    return (
        2
        * precision
        * recall
        / (
            precision
            + recall
        )
    )


token_f1(
    "prevent attention to future tokens",
    "prevents the decoder from attending to future tokens",
)

# 11. String Normalization

Normalization decisions affect automatic metrics.

Over-aggressive normalization may erase meaningful distinctions, especially in
multilingual or fully vocalized text.

# 12. Factuality Labels

For this offline benchmark, a prediction is treated as factually correct when its
token F1 with the reference exceeds a threshold.

In [ ]:
def score_model_column(
    frame: pd.DataFrame,
    prediction_column: str,
) -> pd.DataFrame:
    scored = frame.copy()

    scored["exact_match"] = [
        exact_match(
            reference,
            prediction,
        )
        for reference, prediction
        in zip(
            scored["reference"],
            scored[
                prediction_column
            ],
        )
    ]

    scored["token_f1"] = [
        token_f1(
            reference,
            prediction,
        )
        for reference, prediction
        in zip(
            scored["reference"],
            scored[
                prediction_column
            ],
        )
    ]

    scored["correct"] = (
        scored[
            "token_f1"
        ] >= 0.55
    ).astype(int)

    return scored


model_a_scores = score_model_column(
    evaluation_data,
    "model_a",
)

model_b_scores = score_model_column(
    evaluation_data,
    "model_b",
)

model_a_scores[
    ["id", "token_f1", "correct"]
]

# 13. Groundedness Labels

Groundedness can be approximated by measuring whether the answer's content appears
in the supplied evidence.

In [ ]:
def support_overlap(
    evidence: str,
    prediction: str,
) -> float:
    evidence_tokens = set(
        normalize_text(
            evidence
        ).split()
    )

    prediction_tokens = normalize_text(
        prediction
    ).split()

    content_tokens = [
        token
        for token in prediction_tokens
        if len(token) > 2
    ]

    if not content_tokens:
        return 0.0

    supported = sum(
        token in evidence_tokens
        for token in content_tokens
    )

    return (
        supported
        / len(
            content_tokens
        )
    )


model_a_scores[
    "groundedness"
] = [
    support_overlap(
        evidence,
        prediction,
    )
    for evidence, prediction
    in zip(
        model_a_scores[
            "evidence"
        ],
        model_a_scores[
            "model_a"
        ],
    )
]

model_a_scores[
    ["id", "groundedness"]
]

# 14. Citation Support

Citation support should test whether cited evidence actually supports the claim,
not merely whether a citation is present.

In [ ]:
citation_examples = pd.DataFrame(
    [
        (
            "Recall at k measures whether relevant evidence is retrieved.",
            "Recall at k measures whether relevant evidence appears in the top k.",
            1,
        ),
        (
            "Tokenization guarantees factual answers.",
            "Tokenizer efficiency affects effective context length.",
            0,
        ),
        (
            "Causal masking hides future tokens.",
            "Causal masking prevents attention to future tokens.",
            1,
        ),
    ],
    columns=[
        "claim",
        "cited_evidence",
        "gold_supported",
    ],
)

citation_examples

# 15. Unsupported Claim Detection

In [ ]:
citation_examples[
    "support_score"
] = [
    support_overlap(
        evidence,
        claim,
    )
    for claim, evidence
    in zip(
        citation_examples[
            "claim"
        ],
        citation_examples[
            "cited_evidence"
        ],
    )
]

citation_examples[
    "predicted_supported"
] = (
    citation_examples[
        "support_score"
    ] >= 0.50
).astype(int)

citation_examples

# 16. Hallucination Rate

A simple operational hallucination rate is the proportion of answers below a
groundedness threshold.

In [ ]:
HALLUCINATION_GROUNDEDNESS_THRESHOLD = 0.60

model_a_scores[
    "hallucinated"
] = (
    model_a_scores[
        "groundedness"
    ]
    < HALLUCINATION_GROUNDEDNESS_THRESHOLD
).astype(int)

hallucination_rate = (
    model_a_scores[
        "hallucinated"
    ].mean()
)

hallucination_rate

# 17. Selective Prediction

A reliable system may abstain when confidence is low instead of answering every
question.

# 18. Confidence Scores

Confidence scores can come from model probabilities, self-estimates, ensemble
agreement, or calibrated external models.

They should be validated empirically.

In [ ]:
model_a_scores[
    "confidence"
] = evaluation_data[
    "confidence_a"
]

model_b_scores[
    "confidence"
] = evaluation_data[
    "confidence_b"
]

model_a_scores[
    ["id", "confidence", "correct"]
]

# 19. Calibration

A calibrated model's 80%-confidence predictions should be correct roughly 80% of
the time over many examples.

# 20. Brier Score

For binary correctness:

\[
	ext{Brier} =
\frac{1}{N}
\sum_i (p_i-y_i)^2
\]

In [ ]:
def brier_score(
    confidence,
    correctness,
) -> float:
    confidence = np.asarray(
        confidence,
        dtype=float,
    )

    correctness = np.asarray(
        correctness,
        dtype=float,
    )

    return float(
        np.mean(
            (
                confidence
                - correctness
            ) ** 2
        )
    )


brier_a = brier_score(
    model_a_scores[
        "confidence"
    ],
    model_a_scores[
        "correct"
    ],
)

brier_b = brier_score(
    model_b_scores[
        "confidence"
    ],
    model_b_scores[
        "correct"
    ],
)

pd.Series(
    {
        "Model A Brier": brier_a,
        "Model B Brier": brier_b,
    }
)

# 21. Expected Calibration Error

ECE partitions predictions into confidence bins and compares average confidence
with observed accuracy.

In [ ]:
def expected_calibration_error(
    confidence,
    correctness,
    bin_count: int = 5,
) -> tuple[float, pd.DataFrame]:
    confidence = np.asarray(
        confidence,
        dtype=float,
    )

    correctness = np.asarray(
        correctness,
        dtype=float,
    )

    edges = np.linspace(
        0.0,
        1.0,
        bin_count + 1,
    )

    rows = []
    ece = 0.0

    for bin_index in range(
        bin_count
    ):
        lower = edges[
            bin_index
        ]
        upper = edges[
            bin_index + 1
        ]

        if bin_index == 0:
            mask = (
                confidence >= lower
            ) & (
                confidence <= upper
            )
        else:
            mask = (
                confidence > lower
            ) & (
                confidence <= upper
            )

        count = int(
            mask.sum()
        )

        if count == 0:
            continue

        mean_confidence = float(
            confidence[
                mask
            ].mean()
        )

        accuracy = float(
            correctness[
                mask
            ].mean()
        )

        weight = (
            count
            / len(
                confidence
            )
        )

        ece += (
            weight
            * abs(
                accuracy
                - mean_confidence
            )
        )

        rows.append(
            {
                "bin_lower": lower,
                "bin_upper": upper,
                "count": count,
                "mean_confidence": (
                    mean_confidence
                ),
                "accuracy": accuracy,
            }
        )

    return (
        float(ece),
        pd.DataFrame(rows),
    )


ece_a, reliability_a = (
    expected_calibration_error(
        model_a_scores[
            "confidence"
        ],
        model_a_scores[
            "correct"
        ],
        bin_count=5,
    )
)

print("Model A ECE:", round(ece_a, 4))
reliability_a

# 22. Reliability Diagram

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(
    [0.0, 1.0],
    [0.0, 1.0],
    label="Perfect calibration",
)
plt.plot(
    reliability_a[
        "mean_confidence"
    ],
    reliability_a[
        "accuracy"
    ],
    marker="o",
    label="Model A",
)
plt.xlabel("Mean confidence")
plt.ylabel("Observed accuracy")
plt.title("Reliability Diagram")
plt.legend()
plt.tight_layout()
plt.show()

# 23. Confidence Thresholding

In [ ]:
threshold_rows = []

for threshold in np.linspace(
    0.60,
    0.95,
    8,
):
    selected = model_a_scores[
        model_a_scores[
            "confidence"
        ] >= threshold
    ]

    coverage = (
        len(selected)
        / len(
            model_a_scores
        )
    )

    accuracy = (
        float(
            selected[
                "correct"
            ].mean()
        )
        if len(selected)
        else np.nan
    )

    threshold_rows.append(
        {
            "threshold": threshold,
            "coverage": coverage,
            "selective_accuracy": (
                accuracy
            ),
        }
    )

threshold_frame = pd.DataFrame(
    threshold_rows
)

threshold_frame

# 24. Risk-Coverage Trade-Off

Risk can be defined as:

\[
1 - 	ext{accuracy}
\]

Higher abstention usually lowers coverage and can improve accuracy on answered
examples.

In [ ]:
threshold_frame[
    "risk"
] = (
    1.0
    - threshold_frame[
        "selective_accuracy"
    ]
)

plt.figure(figsize=(7, 5))
plt.plot(
    threshold_frame[
        "coverage"
    ],
    threshold_frame[
        "risk"
    ],
    marker="o",
)
plt.xlabel("Coverage")
plt.ylabel("Risk")
plt.title("Selective Prediction Risk-Coverage")
plt.tight_layout()
plt.show()

# 25. Prompt Robustness

A reliable model should not change its factual answer drastically when the task is
paraphrased without changing meaning.

In [ ]:
paraphrase_groups = [
    {
        "group": "recall",
        "gold": "relevant evidence appears in the top k results",
        "responses": [
            "relevant evidence appears in the top k results",
            "the top k results contain the relevant evidence",
            "it checks whether relevant evidence is among the first k",
        ],
    },
    {
        "group": "context",
        "gold": "maximum tokens processed together",
        "responses": [
            "maximum tokens processed together",
            "the largest token sequence processed at once",
            "how many tokens the model can see together",
        ],
    },
]

paraphrase_groups

# 26. Paraphrase Sensitivity

In [ ]:
robustness_rows = []

for group in paraphrase_groups:
    scores = [
        token_f1(
            group["gold"],
            response,
        )
        for response in group[
            "responses"
        ]
    ]

    robustness_rows.append(
        {
            "group": group["group"],
            "mean_f1": float(
                np.mean(scores)
            ),
            "std_f1": float(
                np.std(scores)
            ),
            "minimum_f1": float(
                np.min(scores)
            ),
        }
    )

pd.DataFrame(
    robustness_rows
)

# 27. Consistency

Consistency measures whether semantically equivalent prompts produce compatible
outputs.

# 28. Adversarial Inputs

Reliability testing should include:

- misleading context;
- irrelevant evidence;
- contradictory evidence;
- instruction-like content in sources;
- spelling variation;
- ambiguous questions.

# 29. Retrieval Versus Generation Errors

In RAG, errors should be attributed to the correct stage.

In [ ]:
rag_error_table = pd.DataFrame(
    [
        ("Retrieval miss", "relevant evidence absent"),
        ("Ranking error", "relevant evidence ranked too low"),
        ("Context selection error", "useful evidence dropped"),
        ("Generation omission", "retrieved evidence ignored"),
        ("Generation hallucination", "unsupported claim added"),
        ("Citation mismatch", "citation does not support claim"),
    ],
    columns=["Error", "Stage interpretation"],
)

rag_error_table

# 30. Citation Precision

Citation precision asks what fraction of cited sources actually support the
associated claims.

In [ ]:
citation_precision = (
    (
        citation_examples[
            "predicted_supported"
        ]
        == 1
    ).sum()
    / len(
        citation_examples
    )
)

citation_precision

# 31. Citation Recall

Citation recall asks what fraction of supportable claims received appropriate
supporting citations.

In [ ]:
gold_supportable = (
    citation_examples[
        "gold_supported"
    ] == 1
)

cited_supported = (
    citation_examples[
        "predicted_supported"
    ] == 1
)

citation_recall = (
    (
        gold_supportable
        & cited_supported
    ).sum()
    / max(
        gold_supportable.sum(),
        1,
    )
)

citation_recall

# 32. Answer Support Rate

In [ ]:
answer_support_rate = (
    model_a_scores[
        "groundedness"
    ] >= 0.60
).mean()

answer_support_rate

# 33. Bootstrap Confidence Intervals

In [ ]:
def bootstrap_mean_ci(
    values,
    iterations: int = 2000,
    confidence_level: float = 0.95,
    seed: int = 42,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    rng = np.random.default_rng(
        seed
    )

    means = []

    for _ in range(
        iterations
    ):
        sample = rng.choice(
            values,
            size=len(values),
            replace=True,
        )

        means.append(
            sample.mean()
        )

    alpha = (
        1.0
        - confidence_level
    )

    lower = np.quantile(
        means,
        alpha / 2
    )

    upper = np.quantile(
        means,
        1.0 - alpha / 2
    )

    return (
        float(
            values.mean()
        ),
        float(lower),
        float(upper),
    )


bootstrap_mean_ci(
    model_a_scores[
        "correct"
    ]
)

# 34. Paired Bootstrap Comparison

Paired resampling preserves example alignment when comparing two models.

In [ ]:
def paired_bootstrap_difference(
    scores_a,
    scores_b,
    iterations: int = 3000,
    seed: int = 42,
):
    scores_a = np.asarray(
        scores_a,
        dtype=float,
    )

    scores_b = np.asarray(
        scores_b,
        dtype=float,
    )

    if len(scores_a) != len(
        scores_b
    ):
        raise ValueError(
            "Paired scores must have the same length."
        )

    rng = np.random.default_rng(
        seed
    )

    differences = []

    indices = np.arange(
        len(scores_a)
    )

    for _ in range(
        iterations
    ):
        sampled_indices = rng.choice(
            indices,
            size=len(indices),
            replace=True,
        )

        difference = (
            scores_a[
                sampled_indices
            ].mean()
            - scores_b[
                sampled_indices
            ].mean()
        )

        differences.append(
            difference
        )

    return {
        "observed_difference": float(
            scores_a.mean()
            - scores_b.mean()
        ),
        "ci_lower": float(
            np.quantile(
                differences,
                0.025,
            )
        ),
        "ci_upper": float(
            np.quantile(
                differences,
                0.975,
            )
        ),
        "probability_a_better": float(
            np.mean(
                np.asarray(
                    differences
                ) > 0
            )
        ),
    }


paired_bootstrap_difference(
    model_a_scores[
        "token_f1"
    ],
    model_b_scores[
        "token_f1"
    ],
)

# 35. Per-Category Analysis

In [ ]:
category_rows = []

for category, group in model_a_scores.groupby(
    "category"
):
    category_rows.append(
        {
            "category": category,
            "examples": len(group),
            "accuracy": group[
                "correct"
            ].mean(),
            "mean_f1": group[
                "token_f1"
            ].mean(),
            "mean_confidence": group[
                "confidence"
            ].mean(),
        }
    )

pd.DataFrame(
    category_rows
)

# 36. Error Taxonomy

In [ ]:
reliability_errors = pd.DataFrame(
    [
        ("Factual error", "claim contradicts reference"),
        ("Unsupported claim", "claim not supported by evidence"),
        ("Omission", "important evidence not used"),
        ("Overconfidence", "high confidence on wrong answer"),
        ("Underconfidence", "low confidence on correct answer"),
        ("Prompt sensitivity", "meaning-equivalent prompts change answer"),
        ("Citation mismatch", "citation does not support claim"),
        ("Abstention failure", "answers when it should defer"),
    ],
    columns=["Error", "Description"],
)

reliability_errors

# 37. Human Evaluation

Human evaluation is essential for qualities that are difficult to capture
automatically.

Typical dimensions:

- correctness;
- completeness;
- groundedness;
- relevance;
- clarity;
- safety.

# 38. Inter-Annotator Agreement

In [ ]:
annotator_one = [
    1, 1, 0, 1, 0, 1, 1, 0, 1, 1
]

annotator_two = [
    1, 1, 0, 0, 0, 1, 1, 0, 1, 1
]

kappa = cohen_kappa_score(
    annotator_one,
    annotator_two,
)

kappa

Agreement should be measured because raw human scores are not automatically
reliable.

# 39. Rubric Design

In [ ]:
human_rubric = pd.DataFrame(
    [
        (
            "Correctness",
            0,
            "factually incorrect",
        ),
        (
            "Correctness",
            1,
            "partly correct",
        ),
        (
            "Correctness",
            2,
            "fully correct",
        ),
        (
            "Groundedness",
            0,
            "unsupported",
        ),
        (
            "Groundedness",
            1,
            "partly supported",
        ),
        (
            "Groundedness",
            2,
            "fully supported",
        ),
    ],
    columns=[
        "Dimension",
        "Score",
        "Description",
    ],
)

human_rubric

# 40. Automatic Judge Models

LLM-as-a-judge evaluation can scale qualitative assessment, but judge models can
introduce:

- positional bias;
- verbosity bias;
- self-preference;
- domain bias;
- prompt sensitivity.

Human validation remains important.

# 41. Benchmark Contamination

If evaluation items appear in training data, measured performance may overestimate
generalization.

Contamination analysis should be part of serious benchmark reporting.

# 42. Dataset Shift

Reliability can degrade when deployment inputs differ from benchmark data.

Evaluate across:

- domains;
- writing styles;
- languages;
- temporal periods;
- adversarial cases.

# 43. Long-Context Reliability

Long contexts introduce new failure modes:

- evidence dilution;
- missed middle-context information;
- conflicting passages;
- citation confusion;
- truncation.

# 44. Safety and Reliability

Safety metrics should be reported separately from ordinary task accuracy.

High capability does not imply safe behavior.

# 45. Multilingual Evaluation

Multilingual LLM evaluation should avoid translating only an English benchmark and
assuming equivalent difficulty.

Native-language evaluation should consider:

- cultural context;
- morphology;
- script;
- tokenizer efficiency;
- annotation quality.

# 46. Arabic Evaluation

Arabic evaluation should explicitly distinguish:

- MSA and dialect;
- vocalized and unvocalized text;
- morphology-sensitive errors;
- clitic segmentation;
- named-entity variation;
- normalization policy.

In [ ]:
arabic_reliability_examples = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "fully vocalized form",
            "exact tashkeel preservation",
        ),
        (
            "بِالْمَدْرَسَةِ",
            "clitic-rich form",
            "morphology-aware matching",
        ),
        (
            "كِتَابُهُمَا",
            "noun plus dual pronoun",
            "agreement and segmentation",
        ),
    ],
    columns=[
        "Form",
        "Property",
        "Evaluation requirement",
    ],
)

arabic_reliability_examples

For fully vocalized Arabic tasks, removing tashkeel during evaluation can hide
meaningful errors. Evaluation should preserve the exact task definition.

# 47. Reproducibility

Record:

- model and revision;
- prompt version;
- decoding settings;
- retrieval settings;
- dataset version;
- normalization rules;
- correctness metric;
- groundedness rubric;
- confidence source;
- calibration procedure;
- bootstrap seed;
- annotator instructions;
- agreement metric;
- known benchmark limitations.

In [ ]:
reproducibility_record = pd.Series(
    {
        "module": (
            "Module 8 • Large Language Models"
        ),
        "lesson": (
            "Lesson 47 • LLM Evaluation, Hallucination, and Reliability"
        ),
        "examples": len(
            evaluation_data
        ),
        "models_compared": 2,
        "correctness_threshold": 0.55,
        "groundedness_threshold": 0.60,
        "bootstrap_seed": 42,
        "offline_execution": True,
        "python": (
            platform.python_version()
        ),
    },
    name="Lesson 47 experiment",
)

reproducibility_record

# 48. Knowledge Check

1. How does reliability differ from capability?
2. What is hallucination?
3. How do factuality and groundedness differ?
4. Why can exact match be too strict?
5. What does token F1 measure?
6. What is a groundedness score?
7. What is calibration?
8. What does the Brier score measure?
9. What is expected calibration error?
10. Why use abstention?
11. What is the risk-coverage trade-off?
12. Why use paired bootstrap comparison?
13. What does citation precision measure?
14. Why measure inter-annotator agreement?
15. Why should multilingual evaluation be language-specific?

# 49. Exercises

## Exercise 1 — Calibration
Add a second confidence source and compare ECE.

## Exercise 2 — Abstention
Choose a threshold that minimizes risk at at least 70% coverage.

## Exercise 3 — Groundedness
Create a claim-level groundedness annotation dataset.

## Exercise 4 — Citation Support
Label 20 answer-citation pairs.

## Exercise 5 — Robustness
Generate ten paraphrases of each evaluation prompt.

## Exercise 6 — Bootstrap
Compare two models with paired bootstrap confidence intervals.

## Exercise 7 — Human Evaluation
Design a three-dimension rubric for correctness, groundedness, and completeness.

## Exercise 8 — Retrieval Attribution
Separate retrieval misses from generation hallucinations.

## Exercise 9 — Arabic Reliability
Build a fully vocalized Arabic evaluation subset.

## Exercise 10 — Evaluation Card
Document metrics, thresholds, confidence intervals, and limitations.

## Challenge Exercises

1. Implement temperature scaling for confidence calibration.
2. Add claim segmentation before groundedness evaluation.
3. Compare human judgments with an automated judge.
4. Build a long-context reliability benchmark.
5. Add multilingual robustness testing across English and Arabic.

# 50. Summary and Next Lesson

In this lesson:

- capability and reliability were separated;
- hallucination, factuality, faithfulness, and groundedness were distinguished;
- exact match and token-level F1 were implemented;
- groundedness and unsupported-claim heuristics were added;
- citation support was evaluated;
- confidence calibration, Brier score, ECE, and reliability diagrams were
  implemented;
- abstention and risk-coverage trade-offs were analyzed;
- paraphrase robustness and consistency were measured;
- retrieval and generation errors were separated;
- bootstrap confidence intervals and paired bootstrap comparison were implemented;
- human evaluation, inter-annotator agreement, benchmark contamination, dataset
  shift, long-context reliability, multilingual evaluation, and Arabic/tashkeel
  considerations were integrated.

## Next Lesson

**Lesson 48: LLM Agents, Tool Use, and Function Calling** introduces tool-augmented
language-model systems, action selection, structured tool arguments, observation
loops, tool-result grounding, failure recovery, and agent evaluation.

# References

- Guo, Z. et al. research on hallucination evaluation in large language models.
- Lin, S. et al. *TruthfulQA: Measuring How Models Mimic Human Falsehoods*.
- Kadavath, S. et al. *Language Models (Mostly) Know What They Know*.
- Liang, P. et al. *Holistic Evaluation of Language Models*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.